# Q4 — Geographic Insights

Where are COVID-19 clinical trials concentrated?  
Which countries specialize in certain therapeutic areas? How is the global landscape distributed?

---

## Setup

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────
# CONFIG — update username if needed
# ─────────────────────────────────────────
DB_USER = "vittoriobariosco"  # your username
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "clinical_trials" # your PostgreSQL database name 
CSV_PATH = "/Users/vittoriobariosco/Documents/APPLICATIONS/MIGx/data/processed/clinical_trials_clean.csv"  #path to cleaned CSV from EDA notebook

# ─────────────────────────────────────────
# 1. CONNECT TO POSTGRESQL
# ─────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


# --- Style Guide ---
COLORS = {
    'primary': '#2563EB',
    'success': '#10B981',
    'danger': '#EF4444',
    'accent': '#F59E0B',
    'secondary': '#64748B',
    'light_gray': '#F1F5F9',
    'dark': '#1E293B'
}

PALETTE = [COLORS['primary'], COLORS['success'], COLORS['danger'],
           COLORS['accent'], COLORS['secondary'], '#8B5CF6', '#EC4899', '#06B6D4']

def style_fig(fig, title='', height=500, width=None, showlegend=True):
    fig.update_layout(
        title=dict(text=title, font=dict(family='Helvetica Neue', size=18, color=COLORS['dark']),
                   x=0, xanchor='left', pad=dict(l=10, t=10)),
        font=dict(family='Helvetica Neue', size=12, color=COLORS['dark']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=height,
        width=width,
        showlegend=showlegend,
        margin=dict(l=60, r=30, t=60, b=50),
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(size=11))
    )
    fig.update_xaxes(showgrid=False, linecolor='#E2E8F0', linewidth=1)
    fig.update_yaxes(showgrid=True, gridcolor='#F1F5F9', linecolor='#E2E8F0', linewidth=1)
    return fig

print('Setup OK — connected to PostgreSQL')

Setup OK — connected to PostgreSQL


---
## Query 4A — Top Countries by Trial Count

**SQL concepts:** 3-table `JOIN` (studies → locations), `COUNT(DISTINCT)` to avoid double-counting multi-site trials, cumulative percentage via window function.

In [2]:
query_4a = """
WITH continent_counts AS (
    SELECT
        l.continent,
        COUNT(DISTINCT l.study_id) AS trial_count
    FROM locations l
    WHERE l.continent IS NOT NULL
    GROUP BY l.continent
),
total AS (
    SELECT COUNT(DISTINCT study_id) AS n FROM locations
    WHERE continent IS NOT NULL
)
SELECT
    c.continent,
    c.trial_count,
    ROUND(100.0 * c.trial_count / t.n, 1) AS pct_of_trials

FROM continent_counts c
CROSS JOIN total t
ORDER BY c.trial_count DESC;
"""

df_4a = pd.read_sql(query_4a, engine)
df_4a

,continent,trial_count,pct_of_trials
0,Europe,2193,42.4
1,North America,1717,33.2
2,Asia,869,16.8
3,South America,347,6.7
4,Africa,290,5.6
5,Other,80,1.5
6,Oceania,52,1.0


In [3]:
# Chart 4A-1: Horizontal bar —
df_plot = df_4a.head(5).sort_values('trial_count', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_plot['continent'], 
    x=df_plot['trial_count'],
    orientation='h',
    marker_color=COLORS['primary'],
    text=df_plot.apply(lambda r: f"{r['trial_count']:,}  ({r['pct_of_trials']}%)", axis=1),
    textposition='outside',
    textfont=dict(size=10)
))

# Set showlegend to False here
style_fig(fig, title='Top continents by Number of Clinical Trials', height=500, width=700, showlegend=False)

fig.update_xaxes(title_text='Number of Trials')
fig.show()

In [4]:
# Chart 4A-3: map
query_map = """
SELECT
    l.country,
    COUNT(DISTINCT s.study_id) AS n_trials
FROM studies s
JOIN locations l ON s.study_id = l.study_id
WHERE l.country IS NOT NULL
GROUP BY l.country;
"""
df_map = pd.read_sql(query_map, engine)

fig = px.choropleth(
    df_map,
    locations='country',
    locationmode='country names',
    color='n_trials',
    color_continuous_scale=[[0, '#EBF2FE'], [0.3, '#93B7F5'], [0.6, '#2563EB'], [1, '#1E3A8A']],
    labels={'n_trials': 'Trials'},
    title='Global Distribution of COVID-19 Clinical Trials'
)

fig.update_layout(
    font=dict(family='Helvetica Neue', size=12, color=COLORS['dark']),
    height=500,
    margin=dict(l=0, r=0, t=50, b=0),
    geo=dict(showframe=False, showcoastlines=True, coastlinecolor='#E2E8F0',
             projection_type='natural earth', bgcolor='white')
)
fig.show()

### Global Distribution of Research Activity

* **Western Centralization**: Europe and North America collectively dominate the research landscape, accounting for **75.6%** of all clinical trials in the dataset.
* **Europe as the Primary Hub**: With **2,193 trials (42.4%)**, Europe maintains the highest concentration of research activity, significantly outpacing other regions.
* **Asia's Mid-Tier Position**: Asia represents the largest footprint outside of the West with **16.8%** of trials, nearly triple the volume of the next closest region, South America.
* **Geographic Disparity**: There is a significant drop-off in research volume across the Global South; South America, Africa, and Oceania combined represent only **13.3%** of the total trial distribution.

---
## Query 4B — Specialization Index by Continent × Therapeutic Area


In [ ]:
query_4c = """
WITH regional AS (
    SELECT
        l.continent,
        c.therapeutic_area,
        COUNT(DISTINCT s.study_id) AS area_trials
    FROM studies s
    JOIN locations l ON s.study_id = l.study_id
    JOIN conditions c ON s.study_id = c.study_id
    WHERE l.continent IS NOT NULL
    GROUP BY l.continent, c.therapeutic_area
),
continent_totals AS (
    SELECT
        l.continent,
        COUNT(DISTINCT l.study_id) AS continent_trials
    FROM locations l
    WHERE l.continent IS NOT NULL
    GROUP BY l.continent
),
global_area AS (
    SELECT
        c.therapeutic_area,
        COUNT(DISTINCT s.study_id) AS global_trials,
        (SELECT COUNT(*) FROM studies) AS total_studies
    FROM studies s
    JOIN conditions c ON s.study_id = c.study_id
    GROUP BY c.therapeutic_area
)

SELECT
    r.continent,
    r.therapeutic_area,
    r.area_trials,

    ROUND(100.0 * r.area_trials / ct.continent_trials, 1) AS local_pct,
    ROUND(100.0 * ga.global_trials / ga.total_studies, 1) AS global_pct,
    ROUND(
        (100.0 * r.area_trials / ct.continent_trials)
        / NULLIF(100.0 * ga.global_trials / ga.total_studies, 0),
        2
    ) AS specialization_index

FROM regional r

JOIN continent_totals ct ON r.continent = ct.continent
JOIN global_area ga ON r.therapeutic_area = ga.therapeutic_area
WHERE r.area_trials >= 5

ORDER BY r.continent, specialization_index DESC;
"""

df_4c = pd.read_sql(query_4c, engine)
df_4c

,continent,therapeutic_area,area_trials,local_pct,global_pct,specialization_index
0,Africa,Infectious Disease,238,82.1,80.0,1.03
1,Africa,Other,73,25.2,26.6,0.95
2,Africa,Metabolic/Cardiovascular,10,3.4,5.3,0.65
3,Africa,Mental Health,12,4.1,7.3,0.57
4,Africa,Respiratory/Critical Care,15,5.2,13.1,0.40
5,Asia,Reproductive Health,13,1.5,0.9,1.65
6,Asia,Other,249,28.7,26.6,1.08
7,Asia,Mental Health,65,7.5,7.3,1.03
8,Asia,Infectious Disease,712,81.9,80.0,1.02
9,Asia,Respiratory/Critical Care,91,10.5,13.1,0.80


In [25]:
# Chart 4C: Heatmap — Specialization index (top 10 countries × therapeutic areas)
# Get top 10 countries by total trial count
top_countries = df_4a.head(10)['continent'].tolist()

df_heat = df_4c[df_4c['continent'].isin(top_countries)].copy()
df_heat_pivot = df_heat.pivot_table(
    index='continent', columns='therapeutic_area',
    values='specialization_index', fill_value=0
)

fig = px.imshow(
    df_heat_pivot.values,
    x=df_heat_pivot.columns.tolist(),
    y=df_heat_pivot.index.tolist(),
    color_continuous_scale=[[0, '#EBF2FE'], [0.5, '#FBBF24'], [1, '#EF4444']],
    aspect='auto',
    text_auto='.2f',
    labels={'color': 'Specialization Index'}
)

style_fig(fig, title='Specialization Index — Top 10 Countries × Therapeutic Area', height=500)
fig.update_xaxes(tickangle=35)
fig.show()

**Key findings:**
The **Specialization Index (SI)** is a way to identify a region's "comparative advantage" in clinical research. It highlights where a continent is focusing more effort than the rest of the world.

### How the Specialization Index is Computed
The SI compares the proportion of trials for a specific therapeutic area within a continent to the proportion of those same trials globally.

$$Specialization Index = \frac{Local\%}{Global\%}$$

* **SI > 1.0**: The region is **more specialized** in this area than the global average.
* **SI = 1.0**: The region’s focus **perfectly matches** the global distribution.
* **SI < 1.0**: The region is **under-represented** in this therapeutic area.

---

### Key Insights from the Specialization Matrix

* **North America's High-Tech Focus**: This region shows the strongest specialization in complex, chronic conditions, specifically **Neurological** ($1.53$) and **Oncology** ($1.40$). This suggests a concentration of high-cost, specialized research infrastructure.
* **Asia's Unique Niche**: Asia stands out with the highest specialization index in the dataset for **Reproductive Health** ($1.65$).
* **Europe's Respiratory Strength**: Europe over-indexes significantly in **Respiratory/Critical Care** ($1.24$) and **Renal** ($1.24$) trials. 
* **South America's Specialization**: Interestingly, South America matches North America's specialization in **Renal** trials ($1.53$) and also shows a strong focus on **Respiratory/Critical Care** ($1.26$).
* **Africa's Infrastructure Gap**: While Africa is specialized in **Infectious Disease** ($1.03$), it has the lowest specialization in **Respiratory/Critical Care** ($0.40$).

---